# Phase 4F: HKO Daily Maximum Temperature

This notebook certifies the Hong Kong Observatory daily maximum
temperature observations used during the two-year weather-only
training period.

It does not access market prices or Polymarket outcomes and does not
fit or select a forecasting model.

In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()

for candidate in (ROOT, *ROOT.parents):
    if (
        candidate
        / "config/v2/hko_daily_max_temperature_spec.json"
    ).exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Repository root not found.")

completed = subprocess.run(
    [
        sys.executable,
        str(
            ROOT
            / "tools/v2/"
            "retrieve_hko_daily_max_temperature.py"
        ),
        "--offline",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(completed.stdout)

if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError("Offline HKO certification failed.")

HKO 2024: CACHED_VALID, rows=366
HKO 2025: CACHED_VALID, rows=365
HKO 2026: CACHED_VALID, rows=181

PHASE 4F HKO TRAINING PANEL COMPLETE
Weather-only training dates: 730
Training interval: 2024-03-16 to 2026-03-15
HKO source years: [2024, 2025, 2026]
Missing HKO training temperatures: 0
Source rows excluded before training: 75
Source rows excluded after training: 107
HKO maximum-temperature range: 14.3 to 35.7 degrees Celsius
Training HKO observations accessed: True
Post-training HKO observations used: 0
Market prices accessed: False
Polymarket outcomes accessed: False
Model fitted or selected: False



In [2]:
import json
import pandas as pd

manifest = json.loads(
    (
        ROOT
        / "data/manifests/v2/"
        "04f_hko_daily_max_manifest.json"
    ).read_text(encoding="utf-8")
)

panel = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "04f_hko_daily_max_training_panel.csv"
)

summary = pd.read_csv(
    ROOT
    / "outputs/v2/final_tables/"
    "04f_hko_daily_max_summary.csv"
)

print("Status:", manifest["status"])
print("Training dates:", len(panel))
print(
    "Training interval:",
    panel["target_date"].min(),
    "to",
    panel["target_date"].max(),
)
print()
print(summary.to_string(index=False))

Status: HKO_DAILY_MAXIMUM_TRAINING_PANEL_CERTIFIED
Training dates: 730
Training interval: 2024-03-16 to 2026-03-15

 weather_only_training_dates first_training_date last_training_date   source_years  minimum_hko_daily_max_c  mean_hko_daily_max_c  maximum_hko_daily_max_c  missing_training_temperatures  source_rows_excluded_before_training  source_rows_excluded_after_training  market_prices_accessed  polymarket_outcomes_accessed  model_fitted  model_selected
                         730          2024-03-16         2026-03-15 2024,2025,2026                     14.3                 27.28                     35.7                              0                                    75                                  107                   False                         False         False           False


In [3]:
assert manifest[
    "status"
] == "HKO_DAILY_MAXIMUM_TRAINING_PANEL_CERTIFIED"

assert len(panel) == 730
assert panel["target_date"].is_unique
assert panel["hko_daily_max_c"].notna().all()

assert manifest["post_training_hko_rows_used"] == 0
assert manifest["market_prices_accessed"] is False
assert manifest["polymarket_outcomes_accessed"] is False
assert manifest["model_fitted"] is False
assert manifest["model_selected"] is False

print("PHASE 4F NOTEBOOK VERIFICATION: PASSED")

PHASE 4F NOTEBOOK VERIFICATION: PASSED
